# Stage 3 - Macro regimes

- **Part 1: the baseline.** Four growth/inflation quadrants with simple rules, and a regression of the stock-bond
  correlation on growth and inflation. This is what PCA and clustering have to beat.
- **Part 2a: principal component analysis.** The 10 indicators reduced to a few factors, refitted every month on an
  expanding window, for real-time and revised data.

- **Input:** `data/processed/` (from `02_clean.ipynb`)
- **Output:** `data/processed/baseline_regimes.csv`, `pca_scores.csv`, `pca_loadings_full.csv`, `pca_stability.csv`
- **Target:** the SPY-TLT correlation over the next 3 months (main) and the next month (robustness)

Run from top to bottom in a fresh kernel.

In [1]:
import numpy as np
import pandas as pd

import config
import helpers

## 1. Load the data

In [2]:
macro = {
    "realtime": helpers.load_processed("macro_realtime_12m", index_col="date", parse_dates=True),
    "revised": helpers.load_processed("macro_revised_12m", index_col="date", parse_dates=True),
}
sb = helpers.load_processed("stock_bond_corr", index_col="date", parse_dates=True)

# what the regimes should explain, known only after month t:
# main = correlation over the next 3 months (t+1 to t+3); robustness = next month only (t+1).
# "current" = the same measure over the window that ends at t, already known at t.
TARGETS = {
    "3m": {"next": sb["spy_tlt_corr_3m"].shift(-3, freq="ME"), "current": sb["spy_tlt_corr_3m"]},
    "1m": {"next": sb["spy_tlt_corr"].shift(-1, freq="ME"), "current": sb["spy_tlt_corr"]},
}
corr = sb["spy_tlt_corr"]

ANALYSIS_START = "2006-01-31"   # same sample as the portfolios (all 4 ETFs from 2006)
print(macro["realtime"].shape, corr.shape)

(428, 10) (289,)


## 2. Growth and inflation indices

1. Each indicator is standardised with its **past only** (expanding mean and standard deviation, at least 36 months),
   so no future information enters.
2. Unemployment and claims get a minus sign: when they rise, growth is weaker.
3. z-scores are capped at ±3. Without the cap, COVID (claims +315% in May 2020) would dominate every average.
4. Growth index = average of the 7 growth and labour z-scores; inflation index = average of the 3 inflation z-scores.

In [3]:
GROWTH = ["indpro", "tcu", "retail", "housing", "payrolls", "unrate", "claims"]
INFLATION = ["cpi", "core_cpi", "ppi"]
SIGN = {"unrate": -1, "claims": -1}   # higher = weaker growth
MIN_HISTORY = 36
CAP = 3


def expanding_z(df):
    """z-score of each column using only the data up to that month."""
    mean = df.expanding(MIN_HISTORY).mean()
    std = df.expanding(MIN_HISTORY).std()
    return ((df - mean) / std).clip(-CAP, CAP)


indices = {}
for name, df in macro.items():
    z = expanding_z(df)
    for col, s in SIGN.items():
        z[col] = s * z[col]
    indices[name] = pd.DataFrame({"growth": z[GROWTH].mean(axis=1),
                                  "inflation": z[INFLATION].mean(axis=1)})

indices["realtime"].loc[["2008-06-30", "2008-12-31", "2020-04-30", "2022-06-30"]].round(2)

,growth,inflation
date,,
2008-06-30,-1.43,1.31
2008-12-31,-2.67,-1.26
2020-04-30,-1.52,-0.71
2022-06-30,0.75,3.00


## 3. Quadrants

Each month, growth and inflation are "high" when above the median of their own past (expanding, at least 24 months).

In [4]:
QUADRANTS = {(True, False): "goldilocks", (True, True): "overheating",
             (False, True): "stagflation", (False, False): "slowdown"}


def quadrants(idx):
    past_median = idx.expanding(24).median().shift(1)    # median of the past only
    high = idx > past_median
    labels = [QUADRANTS[(g, i)] for g, i in zip(high["growth"], high["inflation"])]
    return pd.Series(labels, index=idx.index).where(past_median.notna().all(axis=1))


regimes = pd.DataFrame({name: quadrants(idx) for name, idx in indices.items()}).loc[ANALYSIS_START:]
print("months per quadrant:")
print(regimes.apply(pd.Series.value_counts))

same = (regimes["realtime"] == regimes["revised"]).mean()
print(f"\nreal time and revised give the same quadrant in {same:.0%} of months")
print("months where they differ:")
print(regimes[regimes["realtime"] != regimes["revised"]])

months per quadrant:
             realtime  revised
goldilocks         53       47
overheating        48       48
slowdown           51       53
stagflation        96      100

real time and revised give the same quadrant in 96% of months
months where they differ:
               realtime      revised
date                                
2006-10-31  overheating  stagflation
2011-03-31   goldilocks  overheating
2014-06-30   goldilocks  overheating
2015-06-30   goldilocks     slowdown
2015-08-31   goldilocks     slowdown
2015-09-30   goldilocks     slowdown
2015-10-31   goldilocks     slowdown
2016-11-30     slowdown  stagflation
2019-04-30     slowdown  stagflation
2022-11-30  overheating  stagflation


## 4. Stock-bond correlation by quadrant

The quadrant known at the end of month t is compared with the correlation **after** t:
over the next 3 months (main) and the next month (robustness). That is what an investor deciding at t cares about.

In [5]:
rows = []
for h, tg in TARGETS.items():
    nxt = tg["next"].reindex(regimes.index)
    for name in regimes:
        df = pd.DataFrame({"quadrant": regimes[name], "next_corr": nxt}).dropna()
        g = df.groupby("quadrant")["next_corr"]
        rows.append(pd.DataFrame({"target": h, "data": name, "months": g.size(), "mean_corr": g.mean(),
                                  "share_positive": g.apply(lambda x: (x > 0).mean())}))
by_quadrant = pd.concat(rows).reset_index().set_index(["target", "data", "quadrant"])
by_quadrant.round(2)

months  mean_corr  share_positive
target data     quadrant                                      
3m     realtime goldilocks       53      -0.42            0.08
                overheating      48      -0.29            0.23
                slowdown         51      -0.33            0.10
                stagflation      93      -0.12            0.39
       revised  goldilocks       47      -0.42            0.09
                overheating      48      -0.32            0.19
                slowdown         53      -0.34            0.09
                stagflation      97      -0.12            0.39
1m     realtime goldilocks       53      -0.42            0.08
                overheating      48      -0.27            0.23
                slowdown         51      -0.33            0.20
                stagflation      95      -0.11            0.39
       revised  goldilocks       47      -0.42            0.09
                overheating      48      -0.29            0.21
                slowdown         53      -0.35            0.17
                stagflation      99      -0.11            0.39

Are the differences real? Regression of the future correlation on two dummies (high growth, high inflation),
with Newey-West standard errors: monthly correlations are autocorrelated, and the 3-month targets of
consecutive months overlap.

In [6]:
for h, tg in TARGETS.items():
    for name in regimes:
        q = regimes[name]
        X = pd.DataFrame({"high_growth": q.isin(["goldilocks", "overheating"]).astype(float),
                          "high_inflation": q.isin(["overheating", "stagflation"]).astype(float)})[q.notna()]
        table, info = helpers.ols_hac(tg["next"].reindex(X.index).rename("next_corr"), X)
        print(f"\ntarget {h}, {name}: R2 = {info['r2']:.3f}, {info['nobs']} months, {info['lags']} lags")
        print(table.round(3))


target 3m, realtime: R2 = 0.140, 245 months, 4 lags
                 coef     se      t      p
const          -0.304  0.049 -6.170  0.000
high_growth    -0.136  0.069 -1.954  0.051
high_inflation  0.170  0.062  2.733  0.006

target 3m, revised: R2 = 0.142, 245 months, 4 lags
                 coef     se      t      p
const          -0.306  0.048 -6.402  0.000
high_growth    -0.146  0.069 -2.115  0.034
high_inflation  0.170  0.061  2.762  0.006

target 1m, realtime: R2 = 0.107, 247 months, 4 lags
                 coef     se      t      p
const          -0.308  0.048 -6.486  0.000
high_growth    -0.132  0.067 -1.963  0.050
high_inflation  0.188  0.063  2.996  0.003

target 1m, revised: R2 = 0.110, 247 months, 4 lags
                 coef     se     t      p
const          -0.318  0.045 -7.09  0.000
high_growth    -0.135  0.068 -2.00  0.045
high_inflation  0.195  0.061  3.23  0.001


## 5. Regression on the indices

Same idea with the continuous indices instead of the quadrants, three models:
- **A**: future correlation on growth and inflation;
- **B**: adds the current correlation. Correlations are persistent, so the question is whether
  macro data explains anything **beyond** what the current correlation already tells;
- **C**: adds the **level** of CPI inflation (12-month change, in %). The indices say whether inflation is high
  *relative to its own past*; the literature suggests the absolute level matters (the stock-bond correlation
  tends to turn positive when inflation is above roughly 2.5-3%). Test decided before looking at its result.

In [7]:
def model_table(X, cpi_level, label):
    """Models A, B and C for both targets. Each cell: coefficient (p-value)."""
    rows = []
    for h, tg in TARGETS.items():
        y = tg["next"].reindex(X.index).rename("next_corr")
        specs = {"A": X,
                 "B": X.join(tg["current"].reindex(X.index).rename("current_corr")),
                 "C": X.join(cpi_level.reindex(X.index).rename("cpi_level"))}
        for m, R in specs.items():
            table, info = helpers.ols_hac(y, R)
            row = {"data": label, "target": h, "model": m, "R2": round(info["r2"], 3), "months": info["nobs"]}
            row.update({v: f"{table.loc[v, 'coef']:.3f} ({table.loc[v, 'p']:.3f})" for v in table.index[1:]})
            rows.append(row)
    return pd.DataFrame(rows).set_index(["data", "target", "model"])


index_results = pd.concat([model_table(indices[name].loc[ANALYSIS_START:], macro[name]["cpi"], name)
                           for name in indices])
index_results

R2  months          growth       inflation  \
data     target model                                                  
realtime 3m     A      0.090     245  -0.047 (0.257)   0.089 (0.006)   
                B      0.440     245  -0.035 (0.145)   0.036 (0.074)   
                C      0.108     245  -0.036 (0.406)  -0.081 (0.486)   
         1m     A      0.055     247  -0.032 (0.461)   0.083 (0.014)   
                B      0.291     247  -0.023 (0.367)   0.043 (0.026)   
                C      0.066     247  -0.022 (0.631)  -0.078 (0.508)   
revised  3m     A      0.091     245  -0.046 (0.273)   0.089 (0.006)   
                B      0.439     245  -0.033 (0.168)   0.035 (0.078)   
                C      0.106     245  -0.035 (0.433)  -0.068 (0.571)   
         1m     A      0.055     247  -0.031 (0.488)   0.083 (0.014)   
                B      0.291     247  -0.022 (0.402)   0.043 (0.028)   
                C      0.065     247  -0.020 (0.669)  -0.070 (0.569)   

                        current_corr      cpi_level  
data     target model                                
realtime 3m     A                NaN            NaN  
                B      0.636 (0.000)            NaN  
                C                NaN  0.107 (0.100)  
         1m     A                NaN            NaN  
                B      0.503 (0.000)            NaN  
                C                NaN  0.102 (0.127)  
revised  3m     A                NaN            NaN  
                B      0.636 (0.000)            NaN  
                C                NaN  0.099 (0.144)  
         1m     A                NaN            NaN  
                B      0.503 (0.000)            NaN  
                C                NaN  0.096 (0.164)

## 6. Save

In [8]:
baseline = pd.concat({f"{name}_{col}": indices[name][col].loc[ANALYSIS_START:]
                      for name in indices for col in ["growth", "inflation"]}, axis=1)
baseline["realtime_quadrant"] = regimes["realtime"]
baseline["revised_quadrant"] = regimes["revised"]
path = helpers.save_processed(baseline, "baseline_regimes")
print(f"saved {path.name}: {baseline.shape[0]} rows x {baseline.shape[1]} columns")

saved baseline_regimes.csv: 248 rows x 6 columns


# Part 2a - Principal component analysis

PCA turns the 10 standardised indicators into a few uncorrelated factors. It is done three ways:

1. **Full sample** - only to describe and name the factors. It uses all data up to 2026, so it is never used for decisions.
2. **Expanding window** - each month, PCA is refitted on the data known up to that month, and the month gets its factor scores.
   These are the scores the clustering and the portfolios will use.
3. For **real-time and revised data**, and for **12-month and 3-month changes** (robustness check).

The input is the same as in the baseline: z-scores computed with past data only, unemployment and claims with a minus sign
(higher = stronger economy), capped at ±3.

In [9]:
from sklearn.decomposition import PCA

macro_3m = {
    "realtime": helpers.load_processed("macro_realtime_3m", index_col="date", parse_dates=True),
    "revised": helpers.load_processed("macro_revised_3m", index_col="date", parse_dates=True),
}


def signed_z(df):
    """Expanding z-scores (past only, capped), with unemployment and claims flipped."""
    z = expanding_z(df)
    for col, s in SIGN.items():
        z[col] = s * z[col]
    return z.dropna()          # PCA needs all 10 indicators in a month


Z = {(name, "12m"): signed_z(macro[name]) for name in macro}
Z.update({(name, "3m"): signed_z(macro_3m[name]) for name in macro_3m})
for key, z in Z.items():
    print(key, "complete months:", z.index[0].date(), "->", z.index[-1].date(), f"({len(z)})")

('realtime', '12m') complete months: 1996-01-31 -> 2026-08-31 (368)
('revised', '12m') complete months: 1996-01-31 -> 2026-08-31 (368)
('realtime', '3m') complete months: 1996-01-31 -> 2026-08-31 (368)
('revised', '3m') complete months: 1996-01-31 -> 2026-08-31 (368)


## 2a.1 Full-sample PCA: how many factors, and what they mean

Rules of thumb for the number of components: keep those with an eigenvalue above 1 (each explains more than one
indicator's worth of variance, i.e. more than 10% here), or enough to explain about 75% of the variance.

In [10]:
N_PC = 3   # components saved; the main analysis uses the first 2


def align_signs(components, columns):
    """PCA signs are arbitrary. Fix them: PC1 positive on payrolls, PC2 positive on CPI, PC3 positive on PPI."""
    anchors = ["payrolls", "cpi", "ppi"]
    comps = components.copy()
    for k in range(min(len(comps), len(anchors))):
        if comps[k, columns.get_loc(anchors[k])] < 0:
            comps[k] = -comps[k]
    return comps


full = {}
for key, z in Z.items():
    pca = PCA().fit(z)
    comps = align_signs(pca.components_, z.columns)
    full[key] = {"pca": pca, "loadings": pd.DataFrame(comps[:N_PC].T, index=z.columns,
                                                      columns=[f"pc{k+1}" for k in range(N_PC)])}

variance = pd.DataFrame({f"{n}_{h}": full[(n, h)]["pca"].explained_variance_ratio_[:5] for n, h in full},
                        index=[f"pc{k+1}" for k in range(5)])
print("share of variance explained:")
print(variance.round(3))
print("\ncumulative:")
print(variance.cumsum().round(2))
print("\neigenvalues, real time 12m:", full[("realtime", "12m")]["pca"].explained_variance_.round(2)[:5])

share of variance explained:
     realtime_12m  revised_12m  realtime_3m  revised_3m
pc1         0.482        0.479        0.354       0.363
pc2         0.212        0.225        0.212       0.223
pc3         0.120        0.106        0.128       0.119
pc4         0.080        0.079        0.097       0.097
pc5         0.039        0.039        0.063       0.059

cumulative:
     realtime_12m  revised_12m  realtime_3m  revised_3m
pc1          0.48         0.48         0.35        0.36
pc2          0.69         0.70         0.57        0.59
pc3          0.81         0.81         0.69        0.70
pc4          0.89         0.89         0.79        0.80
pc5          0.93         0.93         0.85        0.86

eigenvalues, real time 12m: [6.25 2.75 1.56 1.04 0.51]


In [11]:
print("loadings, real time 12m:")
full[("realtime", "12m")]["loadings"].round(2)

loadings, real time 12m:


,pc1,pc2,pc3
indpro,0.36,-0.15,0.02
tcu,0.25,-0.02,-0.40
retail,0.25,-0.30,0.37
housing,0.15,-0.36,0.24
payrolls,0.36,-0.15,-0.37
unrate,0.39,-0.22,-0.14
claims,0.39,-0.25,0.11
cpi,0.34,0.55,0.02
core_cpi,0.19,0.29,-0.45
ppi,0.37,0.49,0.52


## 2a.2 Expanding-window PCA

For each month t from the point where at least 60 complete months are available:
fit PCA on the z-scores up to t, fix the signs, and compute the scores of month t.
Nothing after t is used. The loadings of every refit are kept, to check how stable the factors are.

In [12]:
MIN_PCA_HISTORY = 60


def expanding_pca(z, n=N_PC, min_history=MIN_PCA_HISTORY):
    scores, loadings, explained = {}, {}, {}
    for i in range(min_history, len(z) + 1):
        window = z.iloc[:i]
        t = window.index[-1]
        pca = PCA(n_components=n).fit(window)
        comps = align_signs(pca.components_, z.columns)
        centred = window.iloc[-1] - pca.mean_
        scores[t] = comps @ centred.to_numpy()
        loadings[t] = comps
        explained[t] = pca.explained_variance_ratio_
    cols = [f"pc{k+1}" for k in range(n)]
    return (pd.DataFrame(scores, index=cols).T,
            loadings,
            pd.DataFrame(explained, index=cols).T)


expanding = {key: expanding_pca(z) for key, z in Z.items()}
for key, (sc, _, ex) in expanding.items():
    print(key, "scores from", sc.index[0].date(), "| variance of last fit:", ex.iloc[-1].round(2).to_list())

('realtime', '12m') scores from 2000-12-31 | variance of last fit: [0.48, 0.21, 0.12]
('revised', '12m') scores from 2000-12-31 | variance of last fit: [0.48, 0.22, 0.11]
('realtime', '3m') scores from 2000-12-31 | variance of last fit: [0.35, 0.21, 0.13]
('revised', '3m') scores from 2000-12-31 | variance of last fit: [0.36, 0.22, 0.12]


## 2a.3 Are the factors stable over time?

Similarity between the loadings of each monthly refit and the full-sample loadings (cosine similarity:
1 = same direction, 0 = unrelated). If the factors keep their meaning over time, the values stay close to 1.

In [13]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


stability = {}
for key, (sc, loads, ex) in expanding.items():
    ref = full[key]["loadings"].T.to_numpy()
    stability[key] = pd.DataFrame({t: [cosine(L[k], ref[k]) for k in range(N_PC)] for t, L in loads.items()},
                                  index=[f"pc{k+1}" for k in range(N_PC)]).T

for key, st in stability.items():
    s = st.loc[ANALYSIS_START:]
    print(f"{key}: similarity with full-sample loadings from 2006, min / median:",
          s.min().round(2).to_dict(), "/", s.median().round(2).to_dict())

('realtime', '12m'): similarity with full-sample loadings from 2006, min / median: {'pc1': 0.76, 'pc2': 0.84, 'pc3': 0.77} / {'pc1': 0.96, 'pc2': 0.96, 'pc3': 0.91}
('revised', '12m'): similarity with full-sample loadings from 2006, min / median: {'pc1': 0.53, 'pc2': 0.55, 'pc3': 0.63} / {'pc1': 0.95, 'pc2': 0.94, 'pc3': 0.77}
('realtime', '3m'): similarity with full-sample loadings from 2006, min / median: {'pc1': 0.56, 'pc2': 0.58, 'pc3': 0.82} / {'pc1': 0.96, 'pc2': 0.98, 'pc3': 0.93}
('revised', '3m'): similarity with full-sample loadings from 2006, min / median: {'pc1': -0.26, 'pc2': -0.4, 'pc3': 0.76} / {'pc1': 0.93, 'pc2': 0.95, 'pc3': 0.9}


In [14]:
# how much does hindsight change the scores? expanding-window vs full-sample scores, from 2006
for key, (sc, _, _) in expanding.items():
    z = Z[key].loc[sc.index]
    full_scores = (z - full[key]["pca"].mean_) @ full[key]["loadings"]
    c = [sc[f"pc{k}"].loc[ANALYSIS_START:].corr(full_scores[f"pc{k}"].loc[ANALYSIS_START:]) for k in (1, 2, 3)]
    print(f"{key}: correlation expanding vs full-sample scores (pc1, pc2, pc3):", np.round(c, 2))

('realtime', '12m'): correlation expanding vs full-sample scores (pc1, pc2, pc3): [0.98 0.96 0.91]
('revised', '12m'): correlation expanding vs full-sample scores (pc1, pc2, pc3): [0.98 0.82 0.84]
('realtime', '3m'): correlation expanding vs full-sample scores (pc1, pc2, pc3): [0.94 0.9  0.93]
('revised', '3m'): correlation expanding vs full-sample scores (pc1, pc2, pc3): [0.92 0.61 0.92]


## 2a.4 Real time against revised

Same horizon, two data versions: how close are the factor scores, and when do they differ most?

In [15]:
for h in ["12m", "3m"]:
    a, b = expanding[("realtime", h)][0].loc[ANALYSIS_START:], expanding[("revised", h)][0].loc[ANALYSIS_START:]
    print(f"\n{h}: correlation real time vs revised:", {c: round(a[c].corr(b[c]), 3) for c in a})
    print(f"{h}: mean absolute difference:", (a - b).abs().mean().round(2).to_dict())

diff = (expanding[("realtime", "12m")][0] - expanding[("revised", "12m")][0]).loc[ANALYSIS_START:]
print("\nlargest real time - revised gaps on pc1 (12m):")
print(diff["pc1"].abs().sort_values(ascending=False).head(6).round(2))
print("\npc1 and pc2 in 2008, real time vs revised (12m):")
print(pd.concat({"realtime": expanding[("realtime", "12m")][0], "revised": expanding[("revised", "12m")][0]},
                axis=1).loc["2008-03-31":"2009-03-31", (slice(None), ["pc1", "pc2"])].round(2))


12m: correlation real time vs revised: {'pc1': np.float64(0.937), 'pc2': np.float64(0.882), 'pc3': np.float64(0.96)}
12m: mean absolute difference: {'pc1': 0.37, 'pc2': 0.38, 'pc3': 0.28}

3m: correlation real time vs revised: {'pc1': np.float64(0.861), 'pc2': np.float64(0.481), 'pc3': np.float64(0.96)}
3m: mean absolute difference: {'pc1': 0.48, 'pc2': 0.62, 'pc3': 0.29}

largest real time - revised gaps on pc1 (12m):
2008-09-30    7.55
2008-10-31    7.43
2008-08-31    6.00
2008-07-31    4.20
2008-06-30    2.91
2008-11-30    2.73
Name: pc1, dtype: float64

pc1 and pc2 in 2008, real time vs revised (12m):
           realtime       revised      
                pc1   pc2     pc1   pc2
2008-03-31     1.07  3.13   -0.33  2.97
2008-04-30     1.04  3.96   -0.95  3.43
2008-05-31     0.80  3.73   -1.17  3.43
2008-06-30     0.74  4.49   -2.17  3.86
2008-07-31     1.42  4.79   -2.78  4.18
2008-08-31     1.83  5.34   -4.17  3.86
2008-09-30     1.75  5.94   -5.80  2.54
2008-10-31     0.83  6.32 

## 2a.5 Do the factors explain the stock-bond correlation?

Same three models as in the baseline (section 5), with the first two factors instead of the growth and
inflation indices.

In [16]:
pca_results = pd.concat([
    model_table(expanding[(n, h)][0][["pc1", "pc2"]].loc[ANALYSIS_START:], macro[n]["cpi"], f"{n} {h}")
    for n, h in [("realtime", "12m"), ("revised", "12m"), ("realtime", "3m"), ("revised", "3m")]])
pca_results

R2  months             pc1             pc2  \
data         target model                                                  
realtime 12m 3m     A      0.076     245  -0.005 (0.544)   0.053 (0.060)   
                    B      0.441     245  -0.008 (0.218)   0.024 (0.088)   
                    C      0.174     245  -0.059 (0.000)  -0.054 (0.181)   
             1m     A      0.042     247  -0.002 (0.859)   0.047 (0.107)   
                    B      0.289     247  -0.003 (0.598)   0.025 (0.113)   
                    C      0.106     247  -0.054 (0.001)  -0.058 (0.160)   
revised 12m  3m     A      0.095     245  -0.009 (0.490)   0.062 (0.023)   
                    B      0.439     245  -0.009 (0.227)   0.025 (0.118)   
                    C      0.114     245  -0.020 (0.196)   0.010 (0.842)   
             1m     A      0.063     247  -0.004 (0.746)   0.059 (0.029)   
                    B      0.293     247  -0.005 (0.582)   0.031 (0.050)   
                    C      0.069     247  -0.012 (0.472)   0.024 (0.633)   
realtime 3m  3m     A      0.033     245   0.014 (0.111)   0.033 (0.142)   
                    B      0.427     245   0.004 (0.490)   0.010 (0.462)   
                    C      0.095     245  -0.008 (0.525)   0.007 (0.761)   
             1m     A      0.022     247   0.019 (0.082)   0.027 (0.233)   
                    B      0.281     247   0.008 (0.259)   0.011 (0.379)   
                    C      0.060     247  -0.003 (0.861)   0.003 (0.906)   
revised 3m   3m     A      0.083     245  -0.003 (0.745)   0.065 (0.003)   
                    B      0.428     245   0.002 (0.719)   0.014 (0.439)   
                    C      0.126     245  -0.010 (0.332)   0.047 (0.039)   
             1m     A      0.067     247   0.003 (0.802)   0.067 (0.002)   
                    B      0.291     247   0.003 (0.674)   0.030 (0.028)   
                    C      0.088     247  -0.003 (0.791)   0.051 (0.027)   

                            current_corr      cpi_level  
data         target model                                
realtime 12m 3m     A                NaN            NaN  
                    B      0.642 (0.000)            NaN  
                    C                NaN  0.146 (0.000)  
             1m     A                NaN            NaN  
                    B      0.510 (0.000)            NaN  
                    C                NaN  0.143 (0.000)  
revised 12m  3m     A                NaN            NaN  
                    B      0.635 (0.000)            NaN  
                    C                NaN  0.059 (0.166)  
             1m     A                NaN            NaN  
                    B      0.499 (0.000)            NaN  
                    C                NaN  0.041 (0.362)  
realtime 3m  3m     A                NaN            NaN  
                    B      0.657 (0.000)            NaN  
                    C                NaN  0.053 (0.003)  
             1m     A                NaN            NaN  
                    B      0.520 (0.000)            NaN  
                    C                NaN  0.050 (0.010)  
revised 3m   3m     A                NaN            NaN  
                    B      0.645 (0.000)            NaN  
                    C                NaN  0.039 (0.011)  
             1m     A                NaN            NaN  
                    B      0.496 (0.000)            NaN  
                    C                NaN  0.034 (0.049)

## 2a.6 Save

- `pca_scores.csv`: expanding-window scores, one column per data version, horizon and factor
- `pca_loadings_full.csv`: full-sample loadings (to describe the factors in the paper)
- `pca_stability.csv`: similarity of each monthly refit with the full-sample loadings

In [17]:
scores_out = pd.concat({f"{n}_{h}": expanding[(n, h)][0] for n, h in expanding}, axis=1)
scores_out.columns = [f"{a}_{b}" for a, b in scores_out.columns]
loadings_out = pd.concat({f"{n}_{h}": full[(n, h)]["loadings"] for n, h in full}, axis=1)
loadings_out.columns = [f"{a}_{b}" for a, b in loadings_out.columns]
stability_out = pd.concat({f"{n}_{h}": stability[(n, h)] for n, h in stability}, axis=1)
stability_out.columns = [f"{a}_{b}" for a, b in stability_out.columns]

for df, name in [(scores_out.rename_axis("date"), "pca_scores"),
                 (loadings_out.rename_axis("indicator"), "pca_loadings_full"),
                 (stability_out.rename_axis("date"), "pca_stability")]:
    path = helpers.save_processed(df, name)
    print(f"saved {path.name}: {df.shape[0]} rows x {df.shape[1]} columns")

saved pca_scores.csv: 309 rows x 12 columns
saved pca_loadings_full.csv: 10 rows x 12 columns
saved pca_stability.csv: 309 rows x 12 columns
